In [ ]:
import os
import glob
import re
from time import perf_counter as now

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

# ============================================================
# ROOT MMSeg low-latency
# ============================================================
MMSEG_ROOT = '/mmsegmentation'
os.chdir(MMSEG_ROOT)

from mmengine.config import Config
from mmengine.registry import init_default_scope
from mmengine.runner import load_checkpoint
from mmseg.registry import MODELS
from mmseg.utils import register_all_modules

# ============================================================
# PATHS / CONFIGURATION
# ============================================================

# Test/stateful config for the low-latency model.
CONFIG_PATH = (
    '/home/maximilianovelez/00_openmmlab_respaldo/'
    'probando_low_latency/mmsegmentation/'
    'zmax_configs/04_test_adaptive_scheduler_stateful.py'
)

# Final checkpoint from 03_train_adaptive_scheduler_02pretrain.py.
# If left as None, it tries to resolve automatically from cfg.CKPT_PATH
# and then from cfg.work_dir.
CKPT_PATH = '/mmsegmentation/work_dirs/bisenet_adaptive_scheduler_from_pretrain02/best_cls_acc_cls_top1_iter_16200.pth'
# Manual example:
# CKPT_PATH = (
#     '/home/maximilianovelez/00_openmmlab_respaldo/'
#     'probando_low_latency/mmsegmentation/'
#     'work_dirs/bisenet_adaptive_scheduler_from_pretrain02/'
#     'best_cls_acc_cls_top1_iter_XXXX.pth'
# )

# Input video, the same one used for the realistic DFF setup.
INPUT_VIDEO = (
    '/0_secuencia_paravideo/1a_secuencia2.mp4'
)

DEVICE = 'cuda:0'

# ============================================================
# ADAPTIVE SCHEDULER
# ============================================================
# Decision:
#   dev_pred > ADAPTIVE_TAU  -> FIRE
#   dev_pred <= ADAPTIVE_TAU -> HOLD
ADAPTIVE_TAU = 0.03

# The first frame is always FIRE.
FIRST_FRAME_FIRE = True

# Force FIRE if too many consecutive HOLD frames accumulate.
# None = decide only using dev_pred > tau.
MAX_HOLD_FRAMES = None

# ============================================================
# GPU-only benchmark
# ============================================================
BENCHMARK_STYLE_NO_AUTOCAST = True
torch.set_grad_enabled(False)
torch.backends.cudnn.benchmark = False
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass

WARMUP_FRAMES = 60
MAX_FRAMES = None             # None = use the full video
GPU_SAMPLE_EVERY = 1          # 1 = measure every frame
GPU_SAMPLE_OFFSET = 0
PROCESS_SCALE = 1.0
PREPROCESS_USE_PINNED = True
PRED_MASK_DTYPE = np.uint8

# Export
SAVE_RESULTS = True
RESULTS_DIR = (
    '/mmsegmentation/output/'
    'results_adaptive_scheduler_tau003_gpu_only'
)
SUMMARY_CSV = os.path.join(RESULTS_DIR, 'summary_gpu_only_global.csv')
DETAILS_CSV = os.path.join(RESULTS_DIR, 'details_gpu_only_global.csv')


In [ ]:
# ============================================================
# UTILITIES
# ============================================================
try:
    from mmseg.structures.dual_task_seg_data_sample import DualTaskSegDataSample as _TestDataSample
except Exception:
    from mmseg.structures import SegDataSample as _TestDataSample

from mmengine.structures import LabelData, PixelData


def _label_from_logits(cls_logits):
    """Gets the integer class from logits [B, C] without depending on LabelData."""
    if cls_logits is None:
        return None
    if torch.is_tensor(cls_logits):
        return int(torch.argmax(cls_logits, dim=1)[0].detach().cpu().item())
    return None


def _resolve_ckpt_path(config_path, ckpt_path=None):
    """Resolves the checkpoint.

    Order:
      1) explicit ckpt_path if it exists.
      2) cfg.CKPT_PATH if it exists and the file exists.
      3) search in cfg.work_dir.
      4) search in the default directory from 03.
    """
    cfg = Config.fromfile(config_path)

    if ckpt_path is not None:
        if not os.path.exists(ckpt_path):
            raise FileNotFoundError(f'CKPT_PATH explícito no existe: {ckpt_path}')
        return ckpt_path

    cfg_ckpt = cfg.get('CKPT_PATH', None)
    if cfg_ckpt is not None and os.path.exists(cfg_ckpt):
        return cfg_ckpt

    candidate_dirs = []

    work_dir = cfg.get('work_dir', None)
    if work_dir is not None:
        if not os.path.isabs(work_dir):
            work_dir = os.path.join(MMSEG_ROOT, work_dir)
        candidate_dirs.append(work_dir)

    candidate_dirs.append(
        os.path.join(
            MMSEG_ROOT,
            'work_dirs/bisenet_adaptive_scheduler_from_pretrain02'
        )
    )

    patterns = [
        'best_seg_mIoU*.pth',
        'best_cls_acc_cls_top1*.pth',
        'best*.pth',
        'latest.pth',
        'iter_*.pth',
        '*.pth',
    ]

    candidates = []
    for d in candidate_dirs:
        if d is None or not os.path.isdir(d):
            continue
        for pat in patterns:
            candidates.extend(glob.glob(os.path.join(d, pat)))

    if not candidates:
        raise RuntimeError(
            'No encontré checkpoints. Revisa CKPT_PATH o work_dir. '
            f'cfg.CKPT_PATH={cfg_ckpt}, candidate_dirs={candidate_dirs}'
        )

    def _score(path):
        name = os.path.basename(path)
        pri = 9
        if 'best_seg_mIoU' in name:
            pri = 0
        elif 'best_cls_acc_cls_top1' in name:
            pri = 1
        elif name.startswith('best'):
            pri = 2
        elif name == 'latest.pth':
            pri = 3
        elif name.startswith('iter_'):
            pri = 4
        m = re.search(r'(?:iter_|_iter_)(\d+)', name)
        it = int(m.group(1)) if m else -1
        return (pri, -it, name)

    candidates = sorted(set(candidates), key=_score)
    return candidates[0]


def _patch_init_cfg_for_inference(cfg):
    """Avoids loading the model zoo pretrain when building for test/benchmark.

    The final checkpoint from 03 contains the backbone weights, so
    init_cfg=open-mmlab://resnet50_v1c is not needed during inference.
    """
    try:
        cfg.model.backbone.init_cfg = None
    except Exception:
        pass
    try:
        cfg.model.backbone.backbone_cfg.init_cfg = None
    except Exception:
        pass
    return cfg


class AdaptiveSchedulerStatefulInferencer:
    """GPU-only stateful inference for BiSeNetAdaptiveVideoSegmentor.

    FIRE:
        Spatial Path + Context Path + FFM. Updates the cache.
    HOLD:
        Current Spatial Path + feature_propagation(context_key, spatial_key, spatial_cur) + FFM.

    Decision:
        dev_pred = keyframe_selector(S_key, S_t)
        FIRE if dev_pred > tau.
    """

    def __init__(self, model, tau=0.02, max_hold_frames=None, first_frame_fire=True):
        self.model = model
        self.model.eval()
        self.device = next(self.model.parameters()).device

        self.tau = float(tau)
        self.max_hold_frames = None if max_hold_frames is None else int(max_hold_frames)
        self.first_frame_fire = bool(first_frame_fire)

        # Input size from the stateful config, if available.
        self.input_w = 512
        self.input_h = 512
        try:
            st_cfg = model.cfg.get('adaptive_stateful_cfg', {})
            size = st_cfg.get('input_size', (512, 512))
            self.input_w = int(size[0])
            self.input_h = int(size[1])
        except Exception:
            pass

        cfg_dp = model.cfg.model.get('data_preprocessor', {})
        mean = np.array(cfg_dp.get('mean', [123.675, 116.28, 103.53]), dtype=np.float32)
        std  = np.array(cfg_dp.get('std',  [58.395, 57.12, 57.375]), dtype=np.float32)
        self.bgr_to_rgb = bool(cfg_dp.get('bgr_to_rgb', True))

        if mean.size != 3 or std.size != 3:
            raise ValueError(f'Esperaba mean/std de 3 canales. mean={mean.size}, std={std.size}')

        self.mean = torch.tensor(mean, device=self.device, dtype=torch.float32).view(1, 3, 1, 1)
        self.std  = torch.tensor(std,  device=self.device, dtype=torch.float32).view(1, 3, 1, 1)

        self.use_cuda = (self.device.type == 'cuda')
        self.use_pinned = bool(PREPROCESS_USE_PINNED and self.use_cuda)

        self._resize_hwc = np.empty((self.input_h, self.input_w, 3), dtype=np.uint8)
        self._cpu_hwc_t = None
        self._cpu_hwc_np = None
        self._gpu_hwc_u8 = None
        self._gpu_chw_f32 = None

        if self.use_pinned:
            self._cpu_hwc_t = torch.empty((1, self.input_h, self.input_w, 3), dtype=torch.uint8, pin_memory=True)
            self._cpu_hwc_np = self._cpu_hwc_t[0].numpy()
            self._gpu_hwc_u8 = torch.empty((1, self.input_h, self.input_w, 3), dtype=torch.uint8, device=self.device)
            self._gpu_chw_f32 = torch.empty((1, 3, self.input_h, self.input_w), dtype=torch.float32, device=self.device)

        self.reset_state()

    def reset_state(self):
        self.cache = None
        self.frame_counter = 0
        self.hold_count_since_fire = 0
        self.last_phase = 'INIT'
        self.last_dev_pred = np.nan
        self.last_fire_reason = 'RESET'

    def set_tau(self, tau):
        self.tau = float(tau)

    def _preprocess_single(self, img_bgr_nd):
        if (img_bgr_nd.shape[1], img_bgr_nd.shape[0]) != (self.input_w, self.input_h):
            cv2.resize(img_bgr_nd, (self.input_w, self.input_h), dst=self._resize_hwc, interpolation=cv2.INTER_LINEAR)
            img = self._resize_hwc
        else:
            img = img_bgr_nd

        if self.bgr_to_rgb:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.use_pinned:
            np.copyto(self._cpu_hwc_np, img)
            self._gpu_hwc_u8.copy_(self._cpu_hwc_t, non_blocking=True)
            chw_u8 = self._gpu_hwc_u8.permute(0, 3, 1, 2)
            self._gpu_chw_f32.copy_(chw_u8)
            self._gpu_chw_f32.sub_(self.mean).div_(self.std)
            return self._gpu_chw_f32

        img = np.ascontiguousarray(img.transpose(2, 0, 1))
        tensor = torch.from_numpy(img).unsqueeze(0)
        if self.use_cuda:
            tensor = tensor.to(self.device, non_blocking=True)
        tensor = tensor.float()
        tensor = (tensor - self.mean) / self.std
        return tensor

    def _decode_from_fuse_tensors(self, x_fuse, context8_like, context16_like, spatial_like):
        feats = (x_fuse, context8_like, context16_like, spatial_like)

        seg_logits = self.model.decode_head.forward(feats)
        if seg_logits.shape[-2:] != (self.input_h, self.input_w):
            seg_logits = F.interpolate(
                seg_logits,
                size=(self.input_h, self.input_w),
                mode='bilinear',
                align_corners=False
            )

        seg_pred = seg_logits.argmax(dim=1)[0]  # GPU tensor [H, W]

        cls_logits = None
        if self.model.cls_head is not None:
            cls_logits = self.model.cls_head.forward(x_fuse)

        return seg_pred, cls_logits

    def _run_full_fire(self, inputs, spatial_cur=None):
        x_context8, x_context16 = self.model.backbone.context_path(inputs)
        if spatial_cur is None:
            spatial_cur = self.model.backbone.spatial_path(inputs)
        x_fuse = self.model.backbone.ffm(spatial_cur, x_context8)

        # Detach cache to avoid accumulating graphs or unnecessary references.
        self.cache = dict(
            spatial_key=spatial_cur.detach(),
            context8_key=x_context8.detach(),
            context16_key=x_context16.detach(),
        )

        seg_pred, cls_logits = self._decode_from_fuse_tensors(
            x_fuse, x_context8, x_context16, spatial_cur
        )
        return seg_pred, cls_logits

    def _run_hold_prop(self, spatial_cur):
        context8_prop = self.model.feature_propagation(
            context_key=self.cache['context8_key'],
            spatial_key=self.cache['spatial_key'],
            spatial_cur=spatial_cur
        )
        context16_key = self.cache.get('context16_key', context8_prop)
        x_fuse = self.model.backbone.ffm(spatial_cur, context8_prop)

        seg_pred, cls_logits = self._decode_from_fuse_tensors(
            x_fuse, context8_prop, context16_key, spatial_cur
        )
        return seg_pred, cls_logits

    @torch.inference_mode()
    def predict_from_inputs(self, inputs, idx=None):
        """Runs a stateful inference step.

        inputs already comes preprocessed on GPU.
        Returns GPU tensors and phase metadata.
        """
        dev_pred = np.nan
        do_fire = False
        fire_reason = ''

        if self.cache is None:
            do_fire = True
            fire_reason = 'FIRST_OR_EMPTY_CACHE'
            spatial_cur = None
        else:
            spatial_cur = self.model.backbone.spatial_path(inputs)

            if self.model.keyframe_selector is None:
                dev_tensor = torch.tensor([1.0], device=self.device)
            else:
                dev_tensor = self.model.keyframe_selector(
                    self.cache['spatial_key'],
                    spatial_cur
                )

            # Move to Python is required to decide FIRE/HOLD.
            dev_pred = float(dev_tensor.detach().cpu().flatten()[0].item())

            if dev_pred > self.tau:
                do_fire = True
                fire_reason = 'DEV_GT_TAU'
            elif self.max_hold_frames is not None and self.hold_count_since_fire >= self.max_hold_frames:
                do_fire = True
                fire_reason = 'MAX_HOLD'
            else:
                do_fire = False
                fire_reason = 'DEV_LE_TAU'

        if do_fire:
            seg_pred, cls_logits = self._run_full_fire(
                inputs,
                spatial_cur=spatial_cur if 'spatial_cur' in locals() else None
            )
            self.hold_count_since_fire = 0
            phase_used = 'FIRE'
        else:
            seg_pred, cls_logits = self._run_hold_prop(spatial_cur)
            self.hold_count_since_fire += 1
            phase_used = 'HOLD'

        self.last_phase = phase_used
        self.last_dev_pred = dev_pred
        self.last_fire_reason = fire_reason
        self.frame_counter += 1

        return dict(
            seg_pred=seg_pred,
            cls_logits=cls_logits,
            phase_used=phase_used,
            dev_pred=dev_pred,
            fire_reason=fire_reason,
        )

    @torch.inference_mode()
    def predict_only(self, img_bgr_nd, idx=None):
        cur = self._preprocess_single(img_bgr_nd)
        return self.predict_from_inputs(cur, idx=idx)


def open_video_frames(video_path, max_frames=None):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f'No se pudo abrir el video: {video_path}')

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    limit = total if (max_frames is None) else min(total if total > 0 else max_frames, max_frames)

    idx = 0
    pbar = tqdm(total=(limit if limit else None), desc='Frames', unit='frame')
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if max_frames is not None and idx >= max_frames:
                break
            yield idx, frame
            idx += 1
            pbar.update(1)
    finally:
        pbar.close()
        cap.release()


def maybe_scale_frame(frame):
    if PROCESS_SCALE == 1.0:
        return frame
    return cv2.resize(frame, None, fx=PROCESS_SCALE, fy=PROCESS_SCALE, interpolation=cv2.INTER_AREA)


def build_stateful_pipeline():
    register_all_modules()

    cfg = Config.fromfile(CONFIG_PATH)
    cfg = _patch_init_cfg_for_inference(cfg)

    init_default_scope(cfg.get('default_scope', 'mmseg'))

    resolved_ckpt = _resolve_ckpt_path(CONFIG_PATH, CKPT_PATH)

    model = MODELS.build(cfg.model)
    load_checkpoint(model, resolved_ckpt, map_location='cpu')
    model.cfg = cfg
    model.to(DEVICE)
    model.eval()

    inferencer = AdaptiveSchedulerStatefulInferencer(
        model=model,
        tau=ADAPTIVE_TAU,
        max_hold_frames=MAX_HOLD_FRAMES,
        first_frame_fire=FIRST_FRAME_FIRE,
    )

    return {
        'model': model,
        'inferencer': inferencer,
        'resolved_ckpt': resolved_ckpt,
        'cfg': cfg,
    }


def reset_adaptive_state(pipe):
    pipe['inferencer'].reset_state()
    pipe['inferencer'].set_tau(ADAPTIVE_TAU)


def warmup_adaptive(pipe, max_frames=60):
    reset_adaptive_state(pipe)
    for idx, frame in open_video_frames(INPUT_VIDEO, max_frames=max_frames):
        small = maybe_scale_frame(frame)
        _ = pipe['inferencer'].predict_only(small, idx=idx)
        if torch.cuda.is_available():
            torch.cuda.synchronize()


def _summary_global(records, resolved_ckpt):
    vals = np.asarray([r['latency_ms'] for r in records], dtype=np.float64)
    if vals.size == 0:
        raise RuntimeError('No se registraron muestras GPU-only.')

    n = int(vals.size)
    fires = int(sum(r['phase_used'] == 'FIRE' for r in records))
    holds = int(sum(r['phase_used'] == 'HOLD' for r in records))

    dev_vals = np.asarray(
        [r['dev_pred'] for r in records if np.isfinite(r['dev_pred'])],
        dtype=np.float64
    )

    row = {
        'method': 'bisenet_adaptive_scheduler_stateful_gpu_only_comparable',
        'metric': 'GPU_ONLY_GLOBAL',
        'n_timed_frames': n,
        'mean_ms': float(vals.mean()),
        'median_ms': float(np.median(vals)),
        'p95_ms': float(np.percentile(vals, 95)),
        'min_ms': float(vals.min()),
        'max_ms': float(vals.max()),
        'fps_from_mean': float(1000.0 / vals.mean()),
        'fires': fires,
        'holds': holds,
        'fire_ratio': float(fires / n),
        'hold_ratio': float(holds / n),
        'adaptive_tau': float(ADAPTIVE_TAU),
        'max_hold_frames': MAX_HOLD_FRAMES,
        'mean_dev_pred_nonfirst': (float(dev_vals.mean()) if dev_vals.size else np.nan),
        'median_dev_pred_nonfirst': (float(np.median(dev_vals)) if dev_vals.size else np.nan),
        'p95_dev_pred_nonfirst': (float(np.percentile(dev_vals, 95)) if dev_vals.size else np.nan),
        'video_path': INPUT_VIDEO,
        'config_path': CONFIG_PATH,
        'ckpt_path': resolved_ckpt,
    }
    return pd.DataFrame([row])


def benchmark_adaptive_gpu_only(pipe, max_frames=None, sample_every=1, sample_offset=0):
    if not torch.cuda.is_available():
        raise RuntimeError('GPU-only requiere CUDA disponible.')

    reset_adaptive_state(pipe)
    records = []
    frame_count = 0

    for idx, frame in open_video_frames(INPUT_VIDEO, max_frames=max_frames):
        small = maybe_scale_frame(frame)
        take_sample = ((frame_count + sample_offset) % sample_every) == 0

        # Comparable with the realistic/DFF notebook: preprocessing is outside the measured section.
        cur_inputs = pipe['inferencer']._preprocess_single(small)

        torch.cuda.synchronize()
        ev0 = torch.cuda.Event(enable_timing=True)
        ev1 = torch.cuda.Event(enable_timing=True)
        ev0.record()
        out = pipe['inferencer'].predict_from_inputs(cur_inputs, idx=idx)
        ev1.record()
        torch.cuda.synchronize()
        latency_ms = float(ev0.elapsed_time(ev1))

        seg_pred = out['seg_pred'].detach().to(dtype=getattr(torch, str(np.dtype(PRED_MASK_DTYPE).name))).cpu().numpy()

        cls_idx = _label_from_logits(out.get('cls_logits', None))

        if take_sample:
            records.append({
                'frame_idx': int(idx),
                'phase_used': out['phase_used'],
                'fire_reason': out['fire_reason'],
                'dev_pred': float(out['dev_pred']) if np.isfinite(out['dev_pred']) else np.nan,
                'adaptive_tau': float(ADAPTIVE_TAU),
                'latency_ms': latency_ms,
                'pred_h': int(seg_pred.shape[0]),
                'pred_w': int(seg_pred.shape[1]),
                'cls_idx': (None if cls_idx is None else int(cls_idx)),
            })

        frame_count += 1

    summary_df = _summary_global(records, pipe['resolved_ckpt'])
    return records, summary_df


In [ ]:
# ============================================================
# MODEL LOADING
# ============================================================
pipe = build_stateful_pipeline()

print('✅ Modelo Adaptive Scheduler low-latency cargado.')
print(f'Checkpoint usado: {pipe["resolved_ckpt"]}')
print(f'ADAPTIVE_TAU = {ADAPTIVE_TAU}')
print(f'MAX_HOLD_FRAMES = {MAX_HOLD_FRAMES}')
print(f'BENCHMARK_STYLE_NO_AUTOCAST: {BENCHMARK_STYLE_NO_AUTOCAST}')
print(f'TF32 matmul activado: {getattr(torch.backends.cuda.matmul, "allow_tf32", None)}')
print(f'TF32 cuDNN activado: {getattr(torch.backends.cudnn, "allow_tf32", None)}')
print(f'cudnn.benchmark: {torch.backends.cudnn.benchmark}')


In [ ]:
# ============================================================
# WARM-UP
# ============================================================
print('Warm-up Adaptive Scheduler low-latency...')
warmup_adaptive(pipe, max_frames=WARMUP_FRAMES)
print('✅ Warm-up listo.')


In [ ]:
# ============================================================
# GPU-ONLY BENCHMARK (NATURAL GLOBAL FIRE+HOLD AVERAGE)
# ============================================================
records, summary_df = benchmark_adaptive_gpu_only(
    pipe,
    max_frames=MAX_FRAMES,
    sample_every=GPU_SAMPLE_EVERY,
    sample_offset=GPU_SAMPLE_OFFSET,
)

details_df = pd.DataFrame(records)

print('=== RESUMEN GLOBAL GPU-ONLY | ADAPTIVE SCHEDULER LOW-LATENCY FULL SEQ (COMPARABLE) ===')
display(summary_df)

if SAVE_RESULTS:
    os.makedirs(RESULTS_DIR, exist_ok=True)
    summary_df.to_csv(SUMMARY_CSV, index=False)
    details_df.to_csv(DETAILS_CSV, index=False)
    print('CSV resumen :', SUMMARY_CSV)
    print('CSV detalle :', DETAILS_CSV)
